# 05 · Selección del modelo ganador y registro
**Regla de decisión fijada antes de mirar la prueba:** entre los 3 modelos ajustados, son *elegibles* los que quedan dentro de una desviación entre pliegues del mejor PR-AUC en CV. Entre los elegibles gana el de **menor costo de negocio en CV** (desempate: tiempo y explicabilidad). El segundo de esa lista es el *challenger*.
**Hiperparámetros:** si el tuning no mejoró más que la desviación entre pliegues (ver `04`), los finalistas usan sus valores por defecto (más simples y sin el sesgo de elegir el mejor de 30).
El conjunto de prueba se usa **una sola vez** (sección 3), con el ganador y el segundo.
Ejecutar con *Restart* y luego *Run All*.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, mlflow
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier
from src.datos import cargar_obt
from src.pipeline import *
from src.experimento import *
sns.set_theme(style="whitegrid")
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)
X, y = train[FEATURES], train[TARGET]
Xte, yte = test[FEATURES], test[TARGET]            # NO se usa hasta la sección 3
FECHA_CORTE = df[FECHA].max().isoformat()
configurar()
FABRICA = {
    "Embeddings+LogReg":  lambda: construir_pipeline(LogisticRegression(max_iter=1000), texto="embeddings"),
    "LogisticRegression": lambda: construir_pipeline(LogisticRegression(max_iter=1000)),
    "LightGBM":           lambda: construir_pipeline(LGBMClassifier(random_state=SEED, verbose=-1)),
}
FAMILIA = {"Embeddings+LogReg": "estado_del_arte", "LogisticRegression": "lineal_regularizado", "LightGBM": "boosting"}

## 1. Elección con evidencia de CV (sin usar la prueba)

In [ ]:
exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
runs = mlflow.search_runs([exp.experiment_id])
cand = runs[runs["tags.fase"] == "candidato"].set_index("tags.mlflow.runName")
t = cand.loc[[f"candidato-{n}" for n in FABRICA],
             ["metrics.cv_pr_auc_mean","metrics.cv_pr_auc_std","metrics.costo_cv_umbral","metrics.brecha_pr_auc","metrics.segundos_entrenamiento"]]
t.columns = ["pr_auc_cv","std_cv","costo_cv","brecha","seg_entren"]; t.index = [i.replace("candidato-","") for i in t.index]
mejor = t["pr_auc_cv"].idxmax()
elegibles = t[t["pr_auc_cv"] >= t.loc[mejor,"pr_auc_cv"] - t.loc[mejor,"std_cv"]].sort_values("costo_cv")
if len(elegibles) < 2: elegibles = t.sort_values("pr_auc_cv", ascending=False)
CHAMPION, CHALLENGER = elegibles.index[0], elegibles.index[1]
display(t.round(4)); print("Elegibles:", list(elegibles.index)); print("CHAMPION:", CHAMPION, "| CHALLENGER:", CHALLENGER)

### ¿La diferencia entre ganador y segundo supera la variación entre pliegues? (comparación pareada por pliegue)

In [ ]:
cvc = validar_cv(FABRICA[CHAMPION](), X, y); cvh = validar_cv(FABRICA[CHALLENGER](), X, y)
dif = np.array(cvc["va_pr"]) - np.array(cvh["va_pr"])
display(pd.DataFrame({CHAMPION: cvc["va_pr"], CHALLENGER: cvh["va_pr"], "dif": dif}, index=range(1,6)).round(4))
print(f"Dif. media = {dif.mean():.4f} | desv. de la diferencia = {dif.std():.4f} | desv. entre pliegues del ganador = {np.std(cvc['va_pr']):.4f}")
print("¿Supera la variación entre pliegues?", abs(dif.mean()) > np.std(cvc["va_pr"]))

## 2. Corridas finales y Model Registry
Se entrena con el train completo y se evalúa en la prueba (una vez) con el umbral elegido en CV. Se registra `resenas-bancos-ec-modelo` con alias `champion` y `challenger`.

In [ ]:
cli = mlflow.MlflowClient()
try: cli.delete_registered_model(NOMBRE_MODELO)       # evita versiones duplicadas al re-ejecutar
except Exception: pass
limpiar_fase("final")
final = {}
for rol, nombre in [("champion", CHAMPION), ("challenger", CHALLENGER)]:
    final[rol] = correr(f"final-{rol}-{nombre}", FABRICA[nombre](), "final", FAMILIA[nombre], X, y,
                        {"modelo": nombre, "hiperparametros": "por_defecto"}, FECHA_CORTE,
                        tags={"rol": rol, "modelo": nombre}, prueba=(Xte, yte), registrar=NOMBRE_MODELO, alias=rol)
for rol in final:
    mv = cli.get_model_version_by_alias(NOMBRE_MODELO, rol)
    print(rol, "→ versión", mv.version, "| run_id", mv.run_id)

## 3. Resultados en la prueba (hold-out temporal) y métrica de negocio

In [ ]:
cols = ["cv_pr_auc_mean","cv_pr_auc_std","test_pr_auc","test_roc_auc","test_f1","test_precision","test_recall","umbral_aplicado","test_costo","test_costo_050"]
tt = pd.DataFrame([{k: v for k, v in r.items() if k != "_modelo"} for r in final.values()]).set_index("nombre")
display(tt[cols].round(4))

In [ ]:
# Referencias en la prueba (mismo costo: falso negativo = 5, falso positivo = 1)
def ref(nombre, pred):
    return {"modelo": nombre, "costo": costo_negocio(yte, pred), "precision": float(((pred==1)&(yte==1)).sum()/max(pred.sum(),1)), "recall": float(((pred==1)&(yte==1)).sum()/(yte==1).sum())}
filas = [ref("revisar todas (todo = negativa)", np.ones(len(yte), int)),
         ref("no revisar ninguna", np.zeros(len(yte), int)),
         ref("regla de negocio (léxico)", ReglaNegocio().predict(Xte))]
for rol, r in final.items():
    pred = (r["_modelo"].predict_proba(Xte)[:,1] >= r["umbral_aplicado"]).astype(int)
    filas.append(ref(f"{rol}: {r['nombre']}", pred))
display(pd.DataFrame(filas).set_index("modelo").round(4))

### Sensibilidad al costo del error (umbral re-elegido en CV para cada razón FN:FP)

In [ ]:
p_te = final["champion"]["_modelo"].predict_proba(Xte)[:,1]
filas = []
for r in (3, 5, 10):
    u, _ = mejor_umbral(cvc["oof_y"], cvc["oof_p"], c_fn=r)
    pred = (p_te >= u).astype(int)
    filas.append({"FN:FP": f"{r}:1", "umbral": u, "costo modelo": costo_negocio(yte, pred, c_fn=r),
                  "costo revisar todas": costo_negocio(yte, np.ones(len(yte), int), c_fn=r),
                  "costo regla": costo_negocio(yte, ReglaNegocio().predict(Xte), c_fn=r),
                  "costo no revisar": costo_negocio(yte, np.zeros(len(yte), int), c_fn=r)})
display(pd.DataFrame(filas).set_index("FN:FP").round(4))

### Desglose del campeón: por banco y sobre textos no vistos en el train

In [ ]:
r = final["champion"]; pred = (p_te >= r["umbral_aplicado"]).astype(int)
filas = []
for b, g in test.assign(p=p_te, pred=pred).groupby("banco"):
    fila = {"banco": b, "n": len(g), "% negativa real": g[TARGET].mean()*100}
    if g[TARGET].nunique() == 2 and len(g) >= 30:
        fila.update(pr_auc=average_precision_score(g[TARGET], g["p"]), recall=((g.pred==1)&(g[TARGET]==1)).sum()/(g[TARGET]==1).sum(),
                    costo=costo_negocio(g[TARGET], g.pred))
    filas.append(fila)
display(pd.DataFrame(filas).set_index("banco").round(3))
nuevo = ~Xte[TEXTO].isin(set(X[TEXTO])).values
print(f"Textos no vistos en train: {nuevo.mean()*100:.1f}% de la prueba | PR-AUC en ellos: {average_precision_score(yte[nuevo], p_te[nuevo]):.4f} | costo: {costo_negocio(yte[nuevo], pred[nuevo]):.4f}")

## 4. Costo y explicabilidad

In [ ]:
c = tt[["segundos_entrenamiento", "ms_inferencia_por_resena"]].round(4)
display(c)

In [ ]:
# Challenger (LogReg con TF-IDF): palabras que más empujan hacia 'negativa' / 'no negativa'
m = final["challenger"]["_modelo"]
if CHALLENGER == "LogisticRegression":
    coef = pd.Series(m.named_steps["modelo"].coef_[0], index=m.named_steps["prep"].get_feature_names_out()).sort_values()
    print("Más 'no negativa':\n", coef.head(10).round(2).to_string(), "\n\nMás 'negativa':\n", coef.tail(10).round(2).to_string())

In [ ]:
# Campeón: importancia por variable (permutación sobre la prueba, solo para explicar; no se usa para elegir)
pi = permutation_importance(final["champion"]["_modelo"], Xte, yte, scoring="average_precision", n_repeats=3, random_state=SEED)
display(pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(4).rename("caída de PR-AUC al permutar").to_frame())

## 5. Exportar la evidencia: `docs/corridas.csv`

In [ ]:
runs = mlflow.search_runs([exp.experiment_id])
runs.to_csv("/work/docs/corridas.csv", index=False)
print(runs.groupby("tags.fase").size().rename("corridas"))